In [61]:
import polars as pl
from glob import glob
from zoneinfo import ZoneInfo

timezone = "Europe/Vienna"

In [66]:
raw_folder = r"G:\Meine Ablage\Study\Master_Thesis\PracticalWork\tirex_loss\data\home_electricity\raw"
files = glob(raw_folder + "/*.csv")

dfs = []
for file in files:
    df = pl.read_csv(file, try_parse_dates=True, decimal_comma=True, separator=";")
    dfs.append(df)

df = pl.concat(dfs).unique(subset=["Datum"], keep="first").sort("Datum")
df

Datum,kWh,kW,Status,
datetime[μs],f64,f64,str,str
2023-06-11 00:00:00,0.077,0.308,"""VALID""",null
2023-06-11 00:15:00,0.035,0.14,"""VALID""",null
2023-06-11 00:30:00,0.035,0.14,"""VALID""",null
2023-06-11 00:45:00,0.035,0.14,"""VALID""",null
2023-06-11 01:00:00,0.037,0.148,"""VALID""",null
…,…,…,…,…
2026-04-25 22:45:00,0.164,0.656,"""VALID""",null
2026-04-25 23:00:00,0.134,0.536,"""VALID""",null
2026-04-25 23:15:00,0.098,0.392,"""VALID""",null


In [67]:
duplicates = df.filter(pl.col('Datum').is_duplicated())
duplicates

Datum,kWh,kW,Status,
datetime[μs],f64,f64,str,str


In [68]:
df = df.with_columns(pl.col("Datum").dt.replace_time_zone(timezone, ambiguous="earliest", non_existent="null"))
df

Datum,kWh,kW,Status,
"datetime[μs, Europe/Vienna]",f64,f64,str,str
2023-06-11 00:00:00 CEST,0.077,0.308,"""VALID""",null
2023-06-11 00:15:00 CEST,0.035,0.14,"""VALID""",null
2023-06-11 00:30:00 CEST,0.035,0.14,"""VALID""",null
2023-06-11 00:45:00 CEST,0.035,0.14,"""VALID""",null
2023-06-11 01:00:00 CEST,0.037,0.148,"""VALID""",null
…,…,…,…,…
2026-04-25 22:45:00 CEST,0.164,0.656,"""VALID""",null
2026-04-25 23:00:00 CEST,0.134,0.536,"""VALID""",null
2026-04-25 23:15:00 CEST,0.098,0.392,"""VALID""",null


In [69]:
datetime_range = pl.datetime_range(start=df['Datum'].min(),
                                   end=df['Datum'].max(),
                                    interval='15m',
                                    eager=True).alias("Datum")
datetime_range

Datum
"datetime[μs, Europe/Vienna]"
2023-06-11 00:00:00 CEST
2023-06-11 00:15:00 CEST
2023-06-11 00:30:00 CEST
2023-06-11 00:45:00 CEST
2023-06-11 01:00:00 CEST
…
2026-04-25 22:45:00 CEST
2026-04-25 23:00:00 CEST
2026-04-25 23:15:00 CEST


In [ ]:
final_df = datetime_range.to_frame().join(
    df, on="Datum", how="left"
    ).select(
        pl.col("Datum", "kW")
        ).with_columns(
            pl.col("kW").forward_fill()
            )
final_df

Datum,kW
"datetime[μs, Europe/Vienna]",f64
2023-06-11 00:00:00 CEST,0.308
2023-06-11 00:15:00 CEST,0.14
2023-06-11 00:30:00 CEST,0.14
2023-06-11 00:45:00 CEST,0.14
2023-06-11 01:00:00 CEST,0.148
…,…
2026-04-25 22:45:00 CEST,0.656
2026-04-25 23:00:00 CEST,0.536
2026-04-25 23:15:00 CEST,0.392


In [71]:
final_df.write_csv("home_electricity.csv")